# Project 8 — Local Blog-to-Thread Converter

## Repurpose Content Across Formats

**Goal:** Transform a blog post into multiple output formats:
Twitter thread, LinkedIn post, email newsletter, and bullet summary.

**Stack:** Ollama · LangChain · Jupyter

### What You'll Learn
1. Format-constrained generation
2. Tone and length adaptation
3. Multi-output prompting from a single source

### Prerequisites
- **Ollama** with a chat model (e.g., `qwen3:8b`)

In [ ]:
!pip install -q langchain langchain-ollama

## Step 1 — Verify Ollama Is Running

In [ ]:
import requests
OLLAMA_BASE = "http://localhost:11434"
try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    models = r.json().get("models", [])
    print(f"Ollama running — {len(models)} model(s) available")
    for m in models[:5]: print(f"  - {m['name']}")
except Exception as e:
    print(f"Cannot reach Ollama at {OLLAMA_BASE}: {e}")
    print("Start Ollama first: ollama serve")

## Step 2 — Sample Blog Post

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.3)

blog = """# Why RAG Is the Most Practical LLM Pattern

Retrieval-Augmented Generation (RAG) combines the power of large language
models with external knowledge retrieval. Instead of relying solely on
what the model memorized during training, RAG retrieves relevant documents
and feeds them as context.

Key benefits:
1. Reduces hallucination by grounding answers in real documents
2. Keeps knowledge current without retraining
3. Works with proprietary data that wasn't in training
4. Cheaper than fine-tuning for most use cases

The basic pipeline: chunk documents, create embeddings, store in a vector
database, retrieve top-k chunks at query time, and generate an answer
using the retrieved context.

RAG isn't perfect. It struggles with complex reasoning across many
documents, and retrieval quality is the bottleneck. But for 80% of
enterprise knowledge tasks, it's the best starting point."""
print(f"Blog post: {len(blog)} chars")
print(blog[:200] + "...")

## Step 3 — Twitter/X Thread

In [ ]:
thread_prompt = ChatPromptTemplate.from_messages([
    ("system", "Convert this blog post into a Twitter/X thread. "
     "Rules: 5-8 tweets, each under 280 chars, use emojis sparingly, "
     "number each tweet, make the first tweet a hook."),
    ("human", "{blog}")
])
thread = (thread_prompt | llm | StrOutputParser()).invoke({"blog": blog})
print("📢 TWITTER THREAD:\n")
print(thread)

## Step 4 — LinkedIn Post

In [ ]:
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "Convert this blog into a LinkedIn post. "
     "Rules: professional tone, 150-300 words, include a call-to-action, "
     "use line breaks for readability."),
    ("human", "{blog}")
])
linkedin = (linkedin_prompt | llm | StrOutputParser()).invoke({"blog": blog})
print("💼 LINKEDIN POST:\n")
print(linkedin)

## Step 5 — Email Newsletter

In [ ]:
email_prompt = ChatPromptTemplate.from_messages([
    ("system", "Convert this blog into an email newsletter. "
     "Include: subject line, greeting, 3-paragraph body, CTA, sign-off. "
     "Friendly but informative tone."),
    ("human", "{blog}")
])
email = (email_prompt | llm | StrOutputParser()).invoke({"blog": blog})
print("📧 EMAIL NEWSLETTER:\n")
print(email)

## Step 6 — Bullet Summary

In [ ]:
bullet_prompt = ChatPromptTemplate.from_messages([
    ("system", "Create a concise bullet-point summary. "
     "5-7 bullets, each one sentence, capture the key message."),
    ("human", "{blog}")
])
bullets = (bullet_prompt | llm | StrOutputParser()).invoke({"blog": blog})
print("📝 BULLET SUMMARY:\n")
print(bullets)

## Step 7 — Format Comparison

In [ ]:
print("FORMAT COMPARISON:\n")
for name, text in [("Thread", thread), ("LinkedIn", linkedin),
                    ("Email", email), ("Bullets", bullets)]:
    words = len(text.split())
    print(f"  {name:<12} {words:>4} words  {len(text):>5} chars")

## Limitations

1. **No fact-checking** across format transforms.
2. **Character limits** not strictly enforced by LLM.
3. **Platform nuances** (hashtags, mentions) are simplified.

## What You Learned

| Concept | What We Did |
|---|---|
| **Format-constrained generation** | Twitter, LinkedIn, email |
| **Tone adaptation** | Professional, casual, friendly |
| **Multi-output** | One source → four formats |
| **Comparison** | Word/char counts across formats |